# EXP17: 回撤深度分析

1. 最大回撤时间段识别
2. 回撤恢复时间统计
3. 不同资本下的破产概率
4. 回撤期间交易特征分析

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, C12_KWARGS
import pandas as pd
import numpy as np

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}

baseline = run_variant(data, "Baseline", **LIVE_KWARGS)
trades = baseline['_trades']
pnls = np.array([t.pnl for t in trades])
equity = np.cumsum(pnls)
print(f"Baseline: {len(trades)} trades, PnL=${pnls.sum():.0f}")

## Part 1: 回撤事件分析

In [ ]:
peak = np.maximum.accumulate(equity)
drawdown = equity - peak

# 找所有超过 $200 的回撤事件
dd_events = []
in_dd = False
dd_start = 0
for i in range(len(drawdown)):
    if drawdown[i] < -50 and not in_dd:
        in_dd = True
        dd_start = i
    elif drawdown[i] >= 0 and in_dd:
        in_dd = False
        max_dd = drawdown[dd_start:i].min()
        if max_dd < -200:
            dd_events.append({
                'start_idx': dd_start,
                'end_idx': i,
                'duration_trades': i - dd_start,
                'max_dd': max_dd,
                'start_date': str(trades[dd_start].entry_time.date()),
                'end_date': str(trades[min(i, len(trades)-1)].entry_time.date()),
            })

print(f"=== Drawdown Events > $200 ({len(dd_events)} found) ===")
dd_df = pd.DataFrame(dd_events)
if len(dd_df) > 0:
    print(dd_df.to_string(index=False))
    print(f"\nAvg recovery: {dd_df['duration_trades'].mean():.0f} trades")
    print(f"Max drawdown: ${dd_df['max_dd'].min():.0f}")
    print(f"Longest recovery: {dd_df['duration_trades'].max()} trades")

## Part 2: 不同资本的破产概率

In [ ]:
np.random.seed(42)
n_sims = 5000

print("=== Ruin Probability by Starting Capital ===")
for capital in [1000, 1500, 2000, 2500, 3000, 4000, 5000]:
    ruin_count = 0
    for _ in range(n_sims):
        shuffled = np.random.permutation(pnls)
        eq = capital + np.cumsum(shuffled)
        if eq.min() <= 0:
            ruin_count += 1
    ruin_pct = ruin_count / n_sims * 100
    mc_equity = np.cumsum(np.random.permutation(pnls))
    mc_max_dd = (mc_equity - np.maximum.accumulate(mc_equity)).min()
    print(f"  ${capital:5d}: ruin={ruin_pct:5.1f}%, "
          f"dd/capital={abs(drawdown.min())/capital*100:.1f}%")

## Part 3: 回撤期间交易特征

In [ ]:
# 标记每笔交易是否在回撤期间
in_dd_mask = drawdown < -100

dd_trades = [trades[i] for i in range(len(trades)) if in_dd_mask[i]]
non_dd_trades = [trades[i] for i in range(len(trades)) if not in_dd_mask[i]]

print(f"=== Trades During Drawdown vs Normal ===")
for label, t_list in [('During DD (>$100)', dd_trades), ('Normal', non_dd_trades)]:
    if len(t_list) == 0: continue
    pnl = sum(t.pnl for t in t_list)
    wr = sum(1 for t in t_list if t.pnl > 0) / len(t_list) * 100
    avg = pnl / len(t_list)
    strategies = pd.Series([t.strategy for t in t_list]).value_counts().to_dict()
    directions = pd.Series([t.direction for t in t_list]).value_counts().to_dict()
    print(f"\n  {label}: {len(t_list)} trades, PnL=${pnl:.0f}, Avg=${avg:.2f}, WR={wr:.1f}%")
    print(f"    Strategies: {strategies}")
    print(f"    Directions: {directions}")